In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms, models

# =======================
# CONFIG
# =======================
DATA_ROOT = "data_processed"
NUM_CLASSES = 7
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# =======================
# TRANSFORMS
# =======================
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# =======================
# DATA LOADING
# =======================
def get_test_loader():
    test_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "baseline/test"), transform=val_test_tfms)
    return DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4), test_ds.classes


def get_baseline_loaders():
    train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "baseline/train"), transform=train_tfms)
    val_ds   = datasets.ImageFolder(os.path.join(DATA_ROOT, "baseline/val"), transform=val_test_tfms)

    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
        DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4),
    )


def get_augmented_loaders(aug_folder):
    base_train = datasets.ImageFolder(os.path.join(DATA_ROOT, "baseline/train"), transform=train_tfms)
    base_val   = datasets.ImageFolder(os.path.join(DATA_ROOT, "baseline/val"), transform=train_tfms)
    aug_ds     = datasets.ImageFolder(os.path.join(DATA_ROOT, aug_folder), transform=train_tfms)

    full_dataset = ConcatDataset([base_train, base_val, aug_ds])

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
        DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4),
    )

# =======================
# MODEL FACTORY
# =======================
def get_model(name):
    if name == "resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V1")
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif name == "densenet121":
        model = models.densenet121(weights="IMAGENET1K_V1")
        model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)

    elif name == "mobilenet_v2":
        model = models.mobilenet_v2(weights="IMAGENET1K_V1")
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(weights="IMAGENET1K_V1")
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

    elif name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)

    elif name == "convnext_tiny":
        model = models.convnext_tiny(weights="IMAGENET1K_V1")
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)

    return model.to(DEVICE)

# =======================
# TRAIN FUNCTION
# =======================
def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(EPOCHS):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                preds = model(imgs).argmax(1)
                correct += (preds == labels).sum().item()

        val_acc = correct / len(val_loader.dataset)
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    return model

# =======================
# TEST EVALUATION
# =======================
def evaluate_model(model, test_loader):
    model.eval()
    class_correct = [0] * NUM_CLASSES
    class_total = [0] * NUM_CLASSES

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)

            for i in range(len(labels)):
                class_total[labels[i]] += 1
                if preds[i] == labels[i]:
                    class_correct[labels[i]] += 1

    class_acc = [class_correct[i] / class_total[i] for i in range(NUM_CLASSES)]
    overall_acc = sum(class_correct) / sum(class_total)

    return class_acc, overall_acc

# =======================
# EXPERIMENT LOOP
# =======================
models_to_run = [
    "resnet18", "resnet50", "densenet121",
    "mobilenet_v2", "efficientnet_b0",
    "vgg16", "convnext_tiny"
]

test_loader, class_names = get_test_loader()

for model_name in models_to_run:
    print(f"\n========== {model_name.upper()} ==========")

    # ---- BASELINE ----
    model = get_model(model_name)
    train_loader, val_loader = get_baseline_loaders()
    model = train_model(model, train_loader, val_loader)
    base_class_acc, base_overall = evaluate_model(model, test_loader)

    # ---- cGAN AUG ----
    model = get_model(model_name)
    train_loader, val_loader = get_augmented_loaders("cgan_aug")
    model = train_model(model, train_loader, val_loader)
    cgan_class_acc, cgan_overall = evaluate_model(model, test_loader)

    # ---- cDiff AUG ----
    model = get_model(model_name)
    train_loader, val_loader = get_augmented_loaders("cdiff_aug")
    model = train_model(model, train_loader, val_loader)
    cdiff_class_acc, cdiff_overall = evaluate_model(model, test_loader)

    # ================= CSV CREATION =================
    rows = []
    for i, cname in enumerate(class_names):
        rows.append([
            cname,
            base_class_acc[i],
            cgan_class_acc[i],
            cgan_class_acc[i] - base_class_acc[i],
            cdiff_class_acc[i],
            cdiff_class_acc[i] - base_class_acc[i],
        ])

    rows.append([
        "Overall",
        base_overall,
        cgan_overall,
        cgan_overall - base_overall,
        cdiff_overall,
        cdiff_overall - base_overall,
    ])

    df = pd.DataFrame(rows, columns=[
        "Class",
        "Baseline Acc",
        "cGAN Acc",
        "Δ cGAN",
        "cDiff Acc",
        "Δ cDiff"
    ])

    csv_path = f"{model_name}_augmentation_comparison.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved results to {csv_path}")
